# Data Generation Check — synthetic SaaS financials

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import pandas as pd
import run_pipeline as rp
t = rp.load(); out = rp.run(write=False)
print('tables:', list(t.keys()))

tables: ['dim_date', 'dim_department', 'dim_account', 'fact_actuals', 'fact_budget', 'fact_forecast', 'fact_headcount', 'fact_saas_metrics', 'fact_saas_metrics_budget', 'fact_saas_metrics_forecast']


In [2]:
# Company shape: ending ARR, headcount, ARR/head
op = out['operating_metrics']
saas = out['saas_metrics_summary']
last = op[op['month']=='2025-12-01'].iloc[0]
arr = saas[saas['month']=='2025-12-01'].iloc[0]['ending_arr']
print(f'Ending ARR (Dec-25): ${arr:,.0f}')
print(f'Headcount: {int(last["total_headcount"])}')
print(f'ARR/head: ${last["arr_per_head"]:,.0f}')
print(f'Gross margin: {last["gross_margin"]*100:.1f}%')
print(f'Operating margin: {last["operating_margin"]*100:.1f}%')

Ending ARR (Dec-25): $30,166,368
Headcount: 157
ARR/head: $192,142
Gross margin: 82.0%
Operating margin: -31.0%


In [3]:
# ARR bridge ties every month
s = t['fact_saas_metrics']
chk = (s['starting_arr']+s['new_arr']+s['expansion_arr']-s['contraction_arr']-s['churned_arr']-s['ending_arr']).abs().max()
print('max ARR bridge discrepancy: $%.4f' % chk)

max ARR bridge discrepancy: $0.0000


In [4]:
# Engineered stories visible in FY2025 (marketing over, revenue under)
line = out['variance_detail']
mkt = line[(line['account_id']=='SM_MKT') & (line['month']>='2025-01-01')]
print('Paid Marketing months over budget in FY25:', int((mkt['var_ab_amount']>0).sum()), 'of', len(mkt))
rev = line[(line['account_id']=='REV_SUB') & (line['month']>='2025-06-01')]
print('Subscription revenue under budget H2 FY25:', bool((rev['actual']<rev['budget']).all()))

Paid Marketing months over budget in FY25: 12 of 12
Subscription revenue under budget H2 FY25: True
